In [22]:
!pip install transformers torch pillow openai python-dotenv pandas tqdm


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import os
import json
import base64
import pandas as pd
from tqdm.notebook import tqdm
from PIL import Image

from dotenv import load_dotenv
from openai import OpenAI

from transformers import TrOCRProcessor, VisionEncoderDecoderModel

In [24]:
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

ocr_model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-printed
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [25]:
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv(
    "GOOGLE_APPLICATION_CREDENTIALS"
)


In [26]:
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [27]:
DATASET_DIR = "dataset"   # folder with your images
OUTPUT_DIR = "outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [28]:
def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [29]:
def run_hf_ocr(path):
    try:
        image = Image.open(path).convert("RGB")

        pixel_values = processor(
            images=image,
            return_tensors="pt"
        ).pixel_values

        generated_ids = ocr_model.generate(pixel_values)

        text = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )[0]

        return text.strip()

    except Exception as e:
        return f"OCR_ERROR: {str(e)}"

In [30]:
def analyze_with_openai(path, model="gpt-4o-mini"):
    img_b64 = encode_image(path)

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": PROMPT
                    },
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{img_b64}",
                        "detail": "high"
                    }
                ]
            }
        ]
    )

    return response.output_text

In [31]:
def needs_escalation(ocr_text):
    if "OCR_ERROR" in ocr_text:
        return True

    if len(ocr_text.strip()) < 60:
        return True

    if "\n\n\n" in ocr_text:
        return True

    return False

In [32]:
def process_image(path):
    file_name = os.path.basename(path)

    ocr_text = run_hf_ocr(path)

    try:
        llm_text = analyze_with_openai(path, model="gpt-5-nano")
    except Exception as e:
        llm_text = f"LLM_ERROR: {str(e)}"

    return {
        "file": file_name,
        "ocr_model": "Hugging Face TrOCR",
        "ocr_output": ocr_text,
        "llm_model": "gpt-4o-mini",
        "llm_output": llm_text
    }

In [33]:
files = [
    os.path.join(DATASET_DIR, f)
    for f in os.listdir(DATASET_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
]

print("Total Images:", len(files))

Total Images: 8


In [34]:
PROMPT = """
Analyze this complex document image carefully.

Tasks:
1. Detect multi-column layout and preserve reading order.
2. Extract any table into valid JSON.
3. Transcribe handwritten or slanted text exactly as seen.
4. If text is unclear, mark [unclear].
5. Preserve semantic grouping.

Return output in this JSON format only:

{
  "layout_type": "",
  "columns": [],
  "tables": [],
  "handwriting": [],
  "notes": []
}
"""

In [35]:
results = []

for file in tqdm(files):
    result = process_image(file)
    results.append(result)

  0%|          | 0/8 [00:00<?, ?it/s]

c:\Users\Admin\Desktop\OpenAI-Day-2\llm_env\Lib\site-packages\transformers\generation\utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [36]:
df = pd.DataFrame(results)

csv_path = os.path.join(OUTPUT_DIR, "phase1_results.csv")
df.to_csv(csv_path, index=False)

print("Saved:", csv_path)
df.head()

Saved: outputs\phase1_results.csv


,file,ocr_model,ocr_output,llm_model,llm_output
0,660172199-Os-Gate-Smasher-notes_page-0005.jpg,Hugging Face TrOCR,(RM),gpt-4o-mini,"{\n ""layout_type"": ""Multi-column with diagram..."
1,660172199-Os-Gate-Smasher-notes_page-0008.jpg,Hugging Face TrOCR,ITEM OVER,gpt-4o-mini,"{\n ""layout_type"": ""multi-column header with ..."
2,660172199-Os-Gate-Smasher-notes_page-0009.jpg,Hugging Face TrOCR,ITEM OVER,gpt-4o-mini,"{\n ""layout_type"": ""multi-column header and m..."
3,660172199-Os-Gate-Smasher-notes_page-0015.jpg,Hugging Face TrOCR,ITEM,gpt-4o-mini,"{\n ""layout_type"": ""two_column"",\n ""columns""..."
4,660172199-Os-Gate-Smasher-notes_page-0026.jpg,Hugging Face TrOCR,ITEM,gpt-4o-mini,"{\n ""layout_type"": ""multi-column table"",\n ""..."


In [37]:
df["layout_winner"] = ""
df["table_winner"] = ""
df["handwriting_winner"] = ""
df["overall_winner"] = ""
df["comments"] = ""

df.head()

,file,ocr_model,ocr_output,llm_model,llm_output,layout_winner,table_winner,handwriting_winner,overall_winner,comments
0,660172199-Os-Gate-Smasher-notes_page-0005.jpg,Hugging Face TrOCR,(RM),gpt-4o-mini,"{\n ""layout_type"": ""Multi-column with diagram...",,,,,
1,660172199-Os-Gate-Smasher-notes_page-0008.jpg,Hugging Face TrOCR,ITEM OVER,gpt-4o-mini,"{\n ""layout_type"": ""multi-column header with ...",,,,,
2,660172199-Os-Gate-Smasher-notes_page-0009.jpg,Hugging Face TrOCR,ITEM OVER,gpt-4o-mini,"{\n ""layout_type"": ""multi-column header and m...",,,,,
3,660172199-Os-Gate-Smasher-notes_page-0015.jpg,Hugging Face TrOCR,ITEM,gpt-4o-mini,"{\n ""layout_type"": ""two_column"",\n ""columns""...",,,,,
4,660172199-Os-Gate-Smasher-notes_page-0026.jpg,Hugging Face TrOCR,ITEM,gpt-4o-mini,"{\n ""layout_type"": ""multi-column table"",\n ""...",,,,,


In [38]:
for i in range(len(df)):
    
    ocr = str(df.loc[i, "ocr_output"]).strip()
    llm = str(df.loc[i, "llm_output"]).strip()

    # default
    layout = "LLM"
    table = "LLM"
    handwriting = "LLM"

    # if OCR has much more text, maybe literal extraction better
    if len(ocr) > len(llm) * 1.3:
        handwriting = "OCR"

    # if no JSON/table structure in LLM output
    if "{" not in llm and "[" not in llm:
        table = "OCR"

    overall = "LLM"

    if [layout, table, handwriting].count("OCR") >= 2:
        overall = "OCR"

    df.loc[i, "layout_winner"] = layout
    df.loc[i, "table_winner"] = table
    df.loc[i, "handwriting_winner"] = handwriting
    df.loc[i, "overall_winner"] = overall

df[["file","ocr_output","llm_output","overall_winner"]].head(10)

,file,ocr_output,llm_output,overall_winner
0,660172199-Os-Gate-Smasher-notes_page-0005.jpg,(RM),"{\n ""layout_type"": ""Multi-column with diagram...",LLM
1,660172199-Os-Gate-Smasher-notes_page-0008.jpg,ITEM OVER,"{\n ""layout_type"": ""multi-column header with ...",LLM
2,660172199-Os-Gate-Smasher-notes_page-0009.jpg,ITEM OVER,"{\n ""layout_type"": ""multi-column header and m...",LLM
3,660172199-Os-Gate-Smasher-notes_page-0015.jpg,ITEM,"{\n ""layout_type"": ""two_column"",\n ""columns""...",LLM
4,660172199-Os-Gate-Smasher-notes_page-0026.jpg,ITEM,"{\n ""layout_type"": ""multi-column table"",\n ""...",LLM
5,660172199-Os-Gate-Smasher-notes_page-0037.jpg,IT RM,"{\n ""layout_type"": ""Two-column (left and righ...",LLM
6,WhatsApp Image 2026-04-24 at 12.22.35.jpeg,ITEM,"{\n ""layout_type"": ""complex multi-column (4-c...",LLM
7,WhatsApp Image 2026-04-24 at 12.22.45.jpeg,TOTAL AMOUNT,"{\n ""layout_type"": ""multi-column"",\n ""column...",LLM


In [39]:
row = 0

print("FILE:", df.loc[row, "file"])
print("\n--- OCR OUTPUT ---\n")
print(df.loc[row, "ocr_output"][:3000])

print("\n--- LLM OUTPUT ---\n")
print(df.loc[row, "llm_output"][:3000])

FILE: 660172199-Os-Gate-Smasher-notes_page-0005.jpg

--- OCR OUTPUT ---

(RM)

--- LLM OUTPUT ---

{
  "layout_type": "Multi-column with diagram and two text sections (Processes vs Threads)",
  "columns": [
    {
      "title": "Processes",
      "content": [
        "1) Processes",
        "2) System calls involved in process",
        "3) Different processes have different copies of Data files, code",
        "4) [unclear] switching is shown",
        "5) Blocking a process will not block another",
        "6) Independent",
        "7) Interdependent"
      ]
    },
    {
      "title": "Threads",
      "content": [
        "1) There is no system call involved",
        "2) OS treats different processes threads differently",
        "3) Threads share same copy of code & data",
        "4) Context switching is faster",
        "5) Blocking a thread will block entire process",
        "6) Independent",
        "7) Interdependent"
      ]
    }
  ],
  "tables": [],
  "handwriting": [
  

In [40]:
print("Total Images:", len(df))
print("OCR Model:", df["ocr_model"].iloc[0])
print("LLM Model:", df["llm_model"].iloc[0])

Total Images: 8
OCR Model: Hugging Face TrOCR
LLM Model: gpt-4o-mini


In [41]:
print("Layout Wins")
print(df["layout_winner"].value_counts())

print("\nTable Wins")
print(df["table_winner"].value_counts())

print("\nHandwriting Wins")
print(df["handwriting_winner"].value_counts())

print("\nOverall Wins")
print(df["overall_winner"].value_counts())

Layout Wins
layout_winner
LLM    8
Name: count, dtype: int64

Table Wins
table_winner
LLM    8
Name: count, dtype: int64

Handwriting Wins
handwriting_winner
LLM    8
Name: count, dtype: int64

Overall Wins
overall_winner
LLM    8
Name: count, dtype: int64


In [42]:
for col in ["layout_winner", "table_winner", "handwriting_winner", "overall_winner"]:
    print(f"\n{col}")
    print((df[col].value_counts(normalize=True) * 100).round(2))


layout_winner
layout_winner
LLM    100.0
Name: proportion, dtype: float64

table_winner
table_winner
LLM    100.0
Name: proportion, dtype: float64

handwriting_winner
handwriting_winner
LLM    100.0
Name: proportion, dtype: float64

overall_winner
overall_winner
LLM    100.0
Name: proportion, dtype: float64


In [43]:
final_path = os.path.join(OUTPUT_DIR, "phase1_scored_results.csv")
df.to_csv(final_path, index=False)

print("Saved reviewed file:", final_path)

Saved reviewed file: outputs\phase1_scored_results.csv
